In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from function.library import *

In [2]:

print(sys.executable)

c:\Users\QIUQIN_\anaconda3\envs\fyp\python.exe


In [3]:
data = pd.read_csv('../data/train.csv')
data.head()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
0,0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0


In [5]:
data = data.drop(columns=['id'])

X = data.drop(columns=['loan_paid_back'])
y = data['loan_paid_back']

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Training Subset Features: {X_train.shape[0]} rows") 
print(f"Test Subset Features: {X_test.shape[0]} rows")

Training Subset Features: 475195 rows
Test Subset Features: 118799 rows


In [ ]:

num_cols = X_train.select_dtypes(include=['number']).columns
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns


In [ ]:

print(X.isnull().sum())

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

annual_income           0
debt_to_income_ratio    0
credit_score            0
loan_amount             0
interest_rate           0
gender                  0
marital_status          0
education_level         0
employment_status       0
loan_purpose            0
grade_subgrade          0
dtype: int64


In [9]:
X.dtypes

annual_income           float64
debt_to_income_ratio    float64
credit_score              int64
loan_amount             float64
interest_rate           float64
gender                   object
marital_status           object
education_level          object
employment_status        object
loan_purpose             object
grade_subgrade           object
dtype: object

In [ ]:

print("Education Level:")
print(X_train['education_level'].unique())

print("Grade Subgrade:")
print(X_train['grade_subgrade'].unique())

Education Level:
["Bachelor's" 'High School' 'Other' "Master's" 'PhD']
Grade Subgrade:
['C5' 'C2' 'D4' 'D2' 'A5' 'F2' 'D1' 'B5' 'E3' 'E2' 'C3' 'D3' 'D5' 'B4'
 'C4' 'E1' 'B2' 'B1' 'C1' 'F4' 'B3' 'F3' 'E5' 'F5' 'A1' 'A2' 'E4' 'F1'
 'A3' 'A4']


In [11]:
#ordinal encoding
education_mapping = {
    'Other': 1,
    'High School': 2,
    "Bachelor's": 3,
    "Master's": 4,
    'PhD': 5
}

X_train['education_level'] = X_train['education_level'].map(education_mapping).fillna(0)
X_test['education_level'] = X_test['education_level'].map(education_mapping).fillna(0)

grade_mapping = {
    'A1': 30, 'A2': 29, 'A3': 28, 'A4': 27, 'A5': 26,
    'B1': 25, 'B2': 24, 'B3': 23, 'B4': 22, 'B5': 21,
    'C1': 20, 'C2': 19, 'C3': 18, 'C4': 17, 'C5': 16,
    'D1': 15, 'D2': 14, 'D3': 13, 'D4': 12, 'D5': 11,
    'E1': 10, 'E2': 9,  'E3': 8,  'E4': 7,  'E5': 6,
    'F1': 5,  'F2': 4,  'F3': 3,  'F4': 2,  'F5': 1
}

X_train['grade_subgrade'] = X_train['grade_subgrade'].map(grade_mapping).fillna(0)
X_test['grade_subgrade'] = X_test['grade_subgrade'].map(grade_mapping).fillna(0)

print("New Education values:", X_train['education_level'].unique())
print("New Grade subgrade values:", X_train['grade_subgrade'].unique())

New Education values: [3 2 1 4 5]
New Grade subgrade values: [16 19 12 14 26  4 15 21  8  9 18 13 11 22 17 10 24 25 20  2 23  3  6  1
 30 29  7  5 28 27]


In [12]:
X_train['gender_Male'] = (X_train['gender'] == 'Male').astype(int)
X_train['gender_Female'] = (X_train['gender'] == 'Female').astype(int)

X_test['gender_Male'] = (X_test['gender'] == 'Male').astype(int)
X_test['gender_Female'] = (X_test['gender'] == 'Female').astype(int)

X_train = X_train.drop(columns=['gender'])
X_test = X_test.drop(columns=['gender'])

In [ ]:
#one-hot encoding
nominal_cols = ['marital_status', 'employment_status', 'loan_purpose']
X_train_encoded = pd.get_dummies(X_train, columns=nominal_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=nominal_cols, drop_first=True)


X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)


In [ ]:

features_to_scale = list(num_cols) + ['education_level', 'grade_subgrade']

scaler = StandardScaler()
X_train_encoded[features_to_scale] = scaler.fit_transform(X_train_encoded[features_to_scale])
X_test_encoded[features_to_scale] = scaler.transform(X_test_encoded[features_to_scale])

In [ ]:

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_encoded, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_resampled.value_counts())

Before SMOTE: loan_paid_back
1.0    379692
0.0     95503
Name: count, dtype: int64
After SMOTE: loan_paid_back
1.0    379692
0.0    379692
Name: count, dtype: int64


In [16]:
joblib.dump(num_imputer, "../models/num_imputer.pkl")
joblib.dump(cat_imputer, "../models/cat_imputer.pkl")
joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']

In [17]:
X_train_resampled.to_csv("../data/X_train.csv", index=False)
X_test_encoded.to_csv("../data/X_test.csv", index=False)
y_train_resampled.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)

In [18]:
data = pd.read_csv("../data/X_train.csv")
data.head()

,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,education_level,grade_subgrade,gender_Male,gender_Female,marital_status_Married,...,employment_status_Self-employed,employment_status_Student,employment_status_Unemployed,loan_purpose_Car,loan_purpose_Debt consolidation,loan_purpose_Education,loan_purpose_Home,loan_purpose_Medical,loan_purpose_Other,loan_purpose_Vacation
0,-1.162487,-0.724888,0.686688,0.596946,-1.517003,0.249440,-0.006168,1,0,True,...,False,False,False,False,True,False,False,False,False,False
1,1.485359,-0.724888,0.849023,-1.104499,-0.207324,-0.965009,0.579902,0,1,False,...,False,False,False,False,True,False,False,False,False,False
2,-0.410816,-0.593724,-0.485733,-1.782185,0.081503,0.249440,-0.787594,0,1,False,...,False,False,False,False,False,False,False,False,True,False
3,-0.837023,-0.768609,0.921172,-0.622986,-0.819835,-2.179457,-0.006168,1,0,True,...,False,False,False,False,False,False,False,False,True,False
4,3.922919,-0.447987,-0.233212,-1.044624,-0.879592,1.463888,-0.396881,0,1,True,...,False,False,True,True,False,False,False,False,False,False
